# Consumir un Claude Managed Agent vía *sessions*

En la lección 2 desplegamos nosotros la API (FastAPI + Docker + fly.io). En esta lección **Anthropic hospeda el agente completo** y nosotros solo lo consumimos a través de **sesiones**: un log de eventos durable que vive en el servidor.

El agente es el **"Analista Económico Chile"** (definido en `agent.py`, en esta misma carpeta):
- **Ejecuta código Python** en un sandbox que solo puede alcanzar [mindicador.cl](https://mindicador.cl/api) → datos duros (UF, dólar, UTM, IPC…).
- Tiene **web_search / web_fetch** habilitados → contexto más amplio (noticias, causas).
- Escribe informes en `/mnt/session/outputs/` → los descargamos por la Files API.

**Requisito:** un `ANTHROPIC_API_KEY` con acceso al beta de Managed Agents en tu `.env` (copia `.env.example`).

> ⚠️ Cada mensaje consume tokens reales — mantén las preguntas cortas y borra la sesión al final.

## 1 · Setup: aprovisionar el agente (idempotente)

Un agente son **3 llamadas de API**: `environments.create` → `agents.create` → `sessions.create`. Las dos primeras se hacen **una sola vez** — `provision()` cachea los ids en `agent_config.json` y las corridas siguientes los reutilizan.

In [1]:
# `agent.py` vive en esta misma carpeta. Aseguramos que el directorio del
# notebook esté en sys.path (Jupyter usa esta carpeta como cwd) e importamos.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import agent

cfg = agent.provision()
cfg

Reusing existing agent (agent_config.json): agent_01H2dqcBDAUetV22c4CXh3kV


{'environment_id': 'env_01NDK9T1h4PQWNN7bERhjqEg',
 'agent_id': 'agent_01H2dqcBDAUetV22c4CXh3kV',
 'model': 'claude-sonnet-5'}

## 2 · Primera sesión: datos duros desde el sandbox

Creamos una sesión (queda **persistida en el servidor**, no en este notebook) y preguntamos por la UF. Fíjate en las líneas `⚙️ [sandbox · ...]`: es el agente **ejecutando Python** contra mindicador.cl — nunca le escribimos una tool `get_uf`, solo le describimos la API en el system prompt.

In [3]:
session_id = agent.start_session(cfg["agent_id"], cfg["environment_id"], title="Demo clase — UF y dólar")
print(f"Sesión creada: {session_id}\n")

_ = agent.print_reply(session_id, "¿Cuál es el valor de la UF hoy? ¿Cuántos pesos son 500 UF?")

Sesión creada: sesn_01CMK8xmyrQwPoWxrW6dzvx2


⚙️  [sandbox · bash]

⚙️  [sandbox · bash]
**Valor de la UF hoy (08-07-2026): $40.842,07**

**500 UF = $20.421.035** (500 × 40.842,07)

Fuente: mindicador.cl (API pública, indicador UF, serie del día).


## 3 · Análisis con contexto: código + web + informe descargable

Ahora un encargo más grande, que combina las dos capacidades: la **serie histórica** (código en el sandbox) y el **por qué** (web_search). Le pedimos además que deje un informe en `/mnt/session/outputs/` para descargarlo.

In [4]:
_ = agent.print_reply(
    session_id,
    "Baja la serie del dólar observado del último año desde mindicador.cl, "
    "calcula estadísticas clave (mín, máx, promedio, variación %), y busca en la web "
    "qué explica los movimientos recientes. Escribe un informe breve en markdown "
    "en /mnt/session/outputs/informe.md y dame un resumen de 3 líneas aquí.",
)


⚙️  [sandbox · bash]

⚙️  [sandbox · bash]
Ahora busco contexto sobre los movimientos recientes del dólar en Chile.
⚙️  [sandbox · web_search]

⚙️  [sandbox · web_search]

⚙️  [sandbox · write]
He generado el informe completo en `/mnt/session/outputs/informe.md`. Resumen:

En los últimos 12 meses (08-07-2025 a 08-07-2026) el dólar osciló entre un **mínimo de $854,25** (10-feb-2026) y un **máximo de $978,07** (31-jul-2025), con un **promedio de $920,06** y una **variación de -1,37%** en el período, cerrando prácticamente estable. Los principales impulsores fueron el **precio del cobre** (factor dominante, correlación -0,70 con el tipo de cambio), la **política monetaria de la Fed y el Banco Central de Chile** (TPM en 4,5%), y episodios geopolíticos como el **conflicto en Medio Oriente**, además de un factor político local por el cambio de gobierno hacia fines de 2025.


In [5]:
# Descargamos lo que el agente escribió en /mnt/session/outputs/ → ./outputs/
archivos = agent.download_outputs(session_id)
archivos

[PosixPath('/Users/josemanuelpenamendez/Documents/GitHub/clases/class_3_5_deployment/outputs/informe.md')]

In [6]:
from IPython.display import Markdown

informe = next((a for a in archivos if a.suffix == ".md"), None)
Markdown(informe.read_text()) if informe else archivos

# Informe: Dólar Observado en Chile — Últimos 12 meses
**Período analizado:** 08-07-2025 a 08-07-2026 (249 observaciones diarias hábiles)
**Fuente de datos duros:** [mindicador.cl](https://mindicador.cl/api/dolar) (API pública del Banco Central de Chile)

---

## 1. Estadísticas clave

| Métrica | Valor | Fecha |
|---|---|---|
| Valor inicial (08-07-2025) | $940,28 | — |
| Valor final (08-07-2026) | $927,36 | — |
| **Mínimo** | **$854,25** | 10-02-2026 |
| **Máximo** | **$978,07** | 31-07-2025 |
| **Promedio del período** | **$920,06** | — |
| **Variación % (12 meses)** | **-1,37%** | — |

El rango de fluctuación del período fue de **$123,82** entre el mínimo y el máximo, equivalente a una banda de volatilidad de aproximadamente 14,5% sobre el valor mínimo. Pese a este rango amplio, el dólar cierra el período prácticamente en el mismo nivel donde partió, con una leve apreciación del peso chileno (-1,37%).

---

## 2. Trayectoria del período

- **Julio-agosto 2025**: el dólar alcanzó su punto más alto del período ($978,07 el 31-07-2025), reflejando los niveles más débiles del peso chileno observados en 2024-2025.
- **Fines de 2025 – enero 2026**: el tipo de cambio se mantuvo en niveles altos, en torno a $900-1.000, en un contexto pre-electoral (balotaje presidencial de diciembre 2025).
- **Febrero 2026**: fuerte apreciación del peso hasta el **mínimo del período, $854,25 (10-02-2026)**.
- **Marzo 2026**: reversión y depreciación hasta cerca de $930-932 a fines de marzo, coincidiendo con un repunte de tensiones geopolíticas.
- **Abril-julio 2026**: estabilización en una banda de $880-950, con el dólar retrocediendo en las últimas semanas hasta $927,36 al 08-07-2026.

---

## 3. Contexto: ¿qué explica los movimientos recientes?

**a) Precio del cobre — el factor dominante.**
<cite index="7-2">El precio del cobre explica 65–70% de los movimientos del tipo de cambio, seguido por la política de la Fed y la demanda china.</cite> <cite index="12-4">El cobre tiene correlación negativa de -0,70 con el USD/CLP: cuando sube el cobre, el peso se aprecia porque mejoran los términos de intercambio. Cada US$0,10 de cambio en el precio del cobre impacta aproximadamente 15-20 pesos en el tipo de cambio de mediano plazo.</cite> Esto es consistente con el mínimo del dólar alcanzado en febrero, período en que <cite index="12-5">el cobre alcanzó y superó los US$5/lb en distintos momentos del primer trimestre de 2026, en un mercado tensionado con baja holgura de oferta, actuando como soporte fundamental para el peso chileno.</cite>

**b) Conflicto en Medio Oriente y petróleo.**
La corrección alcista del dólar entre febrero y marzo de 2026 estuvo ligada a factores externos: <cite index="12-3">una corrección alcista entre fines de febrero y fines de marzo, con el billete verde alcanzando los 932 CLP el 31 de marzo ante el encarecimiento del petróleo derivado del conflicto bélico en Medio Oriente.</cite> Más recientemente, la baja del dólar en julio se ha vinculado en parte a la <cite index="8-4">normalización del tránsito comercial por el Estrecho de Ormuz</cite>, lo que ha aliviado parte de esas presiones.

**c) Política monetaria: Fed y Banco Central de Chile.**
<cite index="12-3">El Banco Central de Chile mantiene su Tasa de Política Monetaria (TPM) en 4,5% desde el recorte unánime de diciembre de 2025.</cite> En el frente externo, datos económicos de EE.UU. han presionado a la baja al dólar: <cite index="8-2">los últimos datos del mercado laboral estadounidense mostraron una creación de empleo más débil de lo esperado, reduciendo las expectativas de que la Reserva Federal deba seguir endureciendo su política monetaria este año, lo que ha llevado al mercado a moderar las probabilidades de un alza de tasas en septiembre, restando impulso al billete verde.</cite>

**d) Factor político local.**
El cambio de gobierno también ha influido en las expectativas de mediano plazo: <cite index="6-2">el giro político hacia la derecha impulsa la confianza empresarial y la inversión privada, dos factores clave para la estabilidad de la moneda local</cite>, en el contexto de que <cite index="7-5">las expectativas de corto plazo están influenciadas por la agenda económica del gobierno de José Antonio Kast, instalado el 11 de marzo de 2026 tras su victoria en el balotaje presidencial del 14 de diciembre de 2025 con el 58,21% de los votos.</cite>

**e) Perspectivas hacia adelante.**
Los analistas de consenso proyectan una banda relativamente estable para lo que resta de 2026: <cite index="7-1">analistas de consenso sitúan el promedio anual 2026 en el rango de 880–930 CLP, con sesgo hacia fortaleza del peso si el cobre se mantiene sobre US$4,95/lb (Cochilco).</cite>

---

## 4. Conclusión

El dólar en Chile mostró en los últimos 12 meses una **volatilidad moderada** (rango de ~$124), impulsada principalmente por el precio internacional del cobre, la política monetaria de la Fed y el Banco Central, y episodios geopolíticos puntuales (Medio Oriente). A pesar de estos vaivenes, el peso chileno cierra el período prácticamente estable, con una leve apreciación de 1,37% frente al valor de hace un año.

*Informe generado automáticamente combinando datos de mindicador.cl (API del Banco Central de Chile) y análisis de prensa especializada.*


## 4 · Sesiones durables: el estado vive en el servidor (la gracia)

En la lección 2 el estado moría con cada request. Acá la sesión es un **log de eventos append-only** en el servidor: podemos listar las sesiones del agente, **reproducir** la conversación completa desde el log, y **reanudarla** — incluso si reinicias el kernel de este notebook o te cambias de computador.

*(Prueba real: reinicia el kernel, corre solo la celda de setup (§1) y luego estas — la conversación sigue ahí.)*

In [7]:
# Todas las sesiones de este agente, persistidas en el servidor
for s in agent.list_sessions(cfg["agent_id"]):
    print(f"{s.created_at:%Y-%m-%d %H:%M}  ·  {s.status:<10}  ·  {s.id}")

2026-07-08 14:37  ·  idle        ·  sesn_01CMK8xmyrQwPoWxrW6dzvx2
2026-07-08 14:33  ·  idle        ·  sesn_01Rh5vpcAD5PyiYhhzSfVvPu
2026-07-08 14:19  ·  idle        ·  sesn_01HZ9ppVLSgJHotrFGiCw6od


In [8]:
# Reproducir la conversación desde el log de eventos (no guardamos nada localmente)
session_id = agent.list_sessions(cfg["agent_id"])[0].id  # la más reciente

for rol, texto in agent.load_history(session_id):
    if rol == "tool":
        print(f"   ⚙️  {texto}")
    else:
        print(f"[{rol}] {texto[:200]}{'…' if len(texto) > 200 else ''}\n")

[user] ¿Cuál es el valor de la UF hoy? ¿Cuántos pesos son 500 UF?

   ⚙️  bash
   ⚙️  bash
[agent] **Valor de la UF hoy (08-07-2026): $40.842,07**

**500 UF = $20.421.035** (500 × 40.842,07)

Fuente: mindicador.cl (API pública, indicador UF, serie del día).

[user] Baja la serie del dólar observado del último año desde mindicador.cl, calcula estadísticas clave (mín, máx, promedio, variación %), y busca en la web qué explica los movimientos recientes. Escribe un …

   ⚙️  bash
   ⚙️  bash
[agent] Ahora busco contexto sobre los movimientos recientes del dólar en Chile.

   ⚙️  web_search
   ⚙️  web_search
   ⚙️  write
[agent] He generado el informe completo en `/mnt/session/outputs/informe.md`. Resumen:

En los últimos 12 meses (08-07-2025 a 08-07-2026) el dólar osciló entre un **mínimo de $854,25** (10-feb-2026) y un **má…



In [9]:
# Reanudar la MISMA sesión con una pregunta de seguimiento:
# el agente recuerda todo el contexto anterior (UF, dólar) sin que se lo repitamos.
_ = agent.print_reply(session_id, "¿Y cómo se compara eso con la UTM de este mes?")


⚙️  [sandbox · bash]

⚙️  [sandbox · bash]
## Comparación: Dólar vs. UTM

| Indicador | Valor actual | Variación 12 meses |
|---|---|---|
| **Dólar observado** | $927,36 (08-07-2026) | **-1,37%** |
| **UTM** | $71.649 (julio 2026) | **+3,96%** |

**Diferencias clave:**

- **Naturaleza distinta**: el dólar es un precio de mercado que fluctúa diariamente según oferta/demanda, cobre, tasas de interés y geopolítica — de ahí su rango amplio (entre $854 y $978 en el año). La UTM, en cambio, es un valor **fijado mensualmente por el SII** en base a la variación del IPC del mes anterior, por lo que su trayectoria es monotónica y mucho más suave: subió cada mes o se mantuvo, sin retrocesos, desde $68.923 (jul-2025) hasta $71.649 (jul-2026).

- **Sentido opuesto**: mientras el dólar se **abarató** ligeramente en el año (-1,37%, el peso se apreció), la UTM **subió** un 3,96% interanual, reflejando que la inflación acumulada en Chile fue positiva pese a la apreciación cambiaria. Esto tiene sentido

## 5 · Limpieza (opcional)

Borramos la sesión de demo. El agente y el ambiente son solo definiciones — no cuestan nada mantenerlos.

In [ ]:
# agent.delete_session(session_id)
# print("Sesión eliminada")